## 1. Install Dependencies

In [ ]:
!pip install ultralytics albumentations geopandas folium supervision shapely rasterio seaborn scikit-learn

## 2. Advanced Imports

In [ ]:
import os
import cv2
import json
import yaml
import torch
import random
import rasterio
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from tqdm import tqdm
from pathlib import Path
from collections import defaultdict
from shapely.geometry import Point

from ultralytics import YOLO

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings('ignore')

## 3. GPU Validation

In [ ]:
print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 4. Real xView Class Mapping

Replace generic classes with meaningful labels.

In [ ]:
XVIEW_CLASSES = {
    11: 'fixed_wing_aircraft',
    12: 'small_aircraft',
    13: 'cargo_plane',
    17: 'passenger_vehicle',
    18: 'small_car',
    19: 'bus',
    20: 'pickup_truck',
    21: 'utility_truck',
    23: 'truck',
    24: 'cargo_truck',
    25: 'truck_tractor',
    26: 'trailer',
    27: 'truck_w_trailer',
    28: 'truck_w_flatbed',
    29: 'engineering_vehicle',
    32: 'excavator',
    33: 'dump_truck',
    34: 'bulldozer',
    36: 'shipping_container',
    40: 'maritime_vessel',
    41: 'motorboat',
    42: 'sailboat',
    44: 'barge',
    45: 'fishing_vessel',
    47: 'helipad',
    49: 'storage_tank',
    52: 'building'
}

## 5. Threat-Oriented Class Selection

Focus only on high-value intelligence targets.

In [ ]:
THREAT_CLASSES = {
    'cargo_plane',
    'truck',
    'cargo_truck',
    'engineering_vehicle',
    'maritime_vessel',
    'storage_tank'
}

## 6. Improved Augmentation Pipeline

In [ ]:
train_transform = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.CLAHE(p=0.3),
    A.GaussNoise(p=0.2),
    A.MotionBlur(p=0.2),
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.1,
        rotate_limit=15,
        p=0.5
    ),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

## 7. Smart Train/Validation/Test Split

In [ ]:
all_images = list(Path('dataset/images').glob('*.jpg'))

train_imgs, temp_imgs = train_test_split(
    all_images,
    test_size=0.2,
    random_state=42
)

val_imgs, test_imgs = train_test_split(
    temp_imgs,
    test_size=0.5,
    random_state=42
)

print(f"Train: {len(train_imgs)}")
print(f"Val: {len(val_imgs)}")
print(f"Test: {len(test_imgs)}")

## 8. Better YOLO Dataset YAML

In [ ]:
yaml_data = {
    'path': 'processed_dataset',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {
        idx: cls_name
        for idx, cls_name in enumerate(sorted(list(THREAT_CLASSES)))
    }
}

with open('processed_dataset/data.yaml', 'w') as f:
    yaml.dump(yaml_data, f)

## 9. Advanced YOLOv8 Training

In [ ]:
model = YOLO('yolov8m.pt')

results = model.train(
    data='processed_dataset/data.yaml',
    epochs=100,
    imgsz=1024,
    batch=8,
    patience=20,
    optimizer='AdamW',
    lr0=0.0005,
    weight_decay=0.0005,
    cos_lr=True,
    device=0,
    workers=4,
    amp=True,
    cache=True,
    mosaic=1.0,
    mixup=0.2,
    project='satellite_threat_detection',
    name='advanced_yolov8'
)

## 10. Evaluation Metrics

In [ ]:
metrics = model.val()

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

## 11. Prediction Visualization

In [ ]:
results = model.predict(
    source='test_images',
    conf=0.4,
    save=True,
    imgsz=1024
)

## 12. Threat Density Heatmap

This converts detections into geospatial threat intelligence.

In [ ]:
threat_points = []

for detection in detections:
    lat = detection['latitude']
    lon = detection['longitude']

    threat_points.append(Point(lon, lat))

geo_df = gpd.GeoDataFrame(
    geometry=threat_points,
    crs='EPSG:4326'
)

geo_df.plot(figsize=(10, 10), alpha=0.6)
plt.title('Threat Density Map')
plt.show()

## 13. Geospatial Intelligence Map

In [ ]:
import folium

m = folium.Map(location=[20, 78], zoom_start=4)

for detection in detections:
    folium.CircleMarker(
        location=[detection['latitude'], detection['longitude']],
        radius=4,
        popup=detection['class_name'],
        color='red'
    ).add_to(m)

m

## 14. Threat Scoring Engine

This converts raw detections into intelligence scoring.

In [ ]:
THREAT_WEIGHTS = {
    'cargo_plane': 10,
    'truck': 6,
    'cargo_truck': 8,
    'engineering_vehicle': 7,
    'maritime_vessel': 9,
    'storage_tank': 5
}


def calculate_threat_score(detections):
    score = 0

    for det in detections:
        score += THREAT_WEIGHTS.get(det['class_name'], 1)

    return score

## 15. Temporal Intelligence

Compare satellite images across time.

In [ ]:
before = cv2.imread('before.png')
after = cv2.imread('after.png')

before_gray = cv2.cvtColor(before, cv2.COLOR_BGR2GRAY)
after_gray = cv2.cvtColor(after, cv2.COLOR_BGR2GRAY)

change = cv2.absdiff(before_gray, after_gray)

_, thresh = cv2.threshold(change, 30, 255, cv2.THRESH_BINARY)

plt.figure(figsize=(12, 6))
plt.imshow(thresh, cmap='gray')
plt.title('Detected Changes Across Time')
plt.show()

## 16. Restricted Zone Monitoring

In [ ]:
RESTRICTED_ZONES = [
    {
        'name': 'Zone_A',
        'lat_min': 18.0,
        'lat_max': 19.0,
        'lon_min': 72.0,
        'lon_max': 73.0
    }
]


def is_inside_zone(lat, lon, zone):
    return (
        zone['lat_min'] <= lat <= zone['lat_max'] and
        zone['lon_min'] <= lon <= zone['lon_max']
    )

## 17. Automated Threat Alert Generation

In [ ]:
for detection in detections:

    for zone in RESTRICTED_ZONES:

        if is_inside_zone(
            detection['latitude'],
            detection['longitude'],
            zone
        ):

            print(
                f"ALERT: {detection['class_name']} detected in {zone['name']}"
            )

## 18. Export Detection Results

In [ ]:
results_df = pd.DataFrame(detections)

results_df.to_csv('threat_detections.csv', index=False)

geo_df.to_file('threat_detections.geojson', driver='GeoJSON')

## 19. Streamlit Dashboard (Optional)

In [ ]:
# app.py

import streamlit as st

st.title('Satellite Threat Intelligence Dashboard')

uploaded = st.file_uploader('Upload Satellite Image')

if uploaded:

    st.image(uploaded)

    results = model.predict(uploaded)

    st.write(results)

## 20. FastAPI Deployment

In [ ]:
from fastapi import FastAPI, UploadFile

app = FastAPI()

model = YOLO('best.pt')

@app.post('/predict')
async def predict(file: UploadFile):

    image_path = file.filename

    results = model.predict(image_path)

    return {
        'detections': str(results)
    }

## 21. Recommended Production Folder Structure

In [1]:
satellite-threat-detection/
│
├── data/
├── notebooks/
├── src/
│   ├── preprocessing/
│   ├── augmentation/
│   ├── training/
│   ├── inference/
│   ├── evaluation/
│   ├── geospatial/
│   ├── temporal/
│   └── deployment/
│
├── configs/
├── models/
├── outputs/
├── dashboards/
├── api/
├── requirements.txt
└── README.md

SyntaxError: invalid character '│' (U+2502) (166115823.py, line 2)